# CyberSentinel-EU Stage 3 Part 2: The RAG Chatbot
**Fatema Husain Hasan (202508958) MSc AI Thesis**

An interactive chatbot over all 3,201 GDPR enforcement records.
Ask questions in plain language; it retrieves relevant cases and answers
with citations, or says "insufficient evidence" when the corpus can't answer.

Also includes the BM25 keyword baseline for comparison (RQ3).

**Setup:** paste Azure key in Cell 2, upload `chroma_gdpr.zip` when prompted (Cell 3).

## 1. Setup

In [1]:
!pip install openai chromadb rank-bm25 --quiet
import pandas as pd
import numpy as np
import json
import os
import textwrap
from openai import AzureOpenAI
import chromadb
print('ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

## 2. Azure credentials

In [2]:
AZURE_ENDPOINT = "https://cybersentinel-resource.openai.azure.com/"
AZURE_KEY = "PASTE_YOUR_KEY"
EMBED_DEPLOYMENT = "text-embedding-3-small"
CHAT_DEPLOYMENT  = "gpt-5-mini"
API_VERSION = "2024-10-21"

client = AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_KEY, api_version=API_VERSION)
print("Azure connected")

Azure connected


## 3. Load the vector store
Upload chroma_gdpr.zip to Github

In [3]:
import zipfile, os
if not os.path.exists('chroma_gdpr'):
    !wget -q https://github.com/Fatimaxx24/202508958_IT9099_Thesis/raw/main/chroma_gdpr.zip
    with zipfile.ZipFile('chroma_gdpr.zip','r') as z:
        z.extractall('.')
    print('vector store downloaded and extracted')

chroma = chromadb.PersistentClient(path="./chroma_gdpr")
col = chroma.get_collection("gdpr_enforcement")
print('records in vector store:', col.count())

vector store downloaded and extracted
records in vector store: 3201


## 4. Load the corpus (for BM25 baseline + metadata)

In [4]:
REPO = "https://raw.githubusercontent.com/Fatimaxx24/202508958_IT9099_Thesis/main/"
df = pd.read_csv(REPO + "gdpr_enforcement_tracker_full.csv", low_memory=False)
df['Summary'] = df['Summary'].astype(str)
df = df[df['Summary'].str.strip().str.len() > 20].reset_index(drop=True)

from rank_bm25 import BM25Okapi
corpus_tokens = [t.lower().split() for t in df['Summary']]
bm25 = BM25Okapi(corpus_tokens)
print('BM25 index built over', len(df), 'records')

BM25 index built over 3201 records


## 5. The RAG chatbot
Ask(question) semantic retrieval + gpt-5-mini answer with citations.

In [6]:
SYSTEM_PROMPT = """You are CyberSentinel-EU, an assistant that answers questions about GDPR
enforcement decisions using only the retrieved records provided.

Rules:
1. Answer only from the retrieved records. Never use outside knowledge.
2. Cite every claim with the record ID in square brackets, e.g. [ETid 2521].
3. If the retrieved records do not contain the answer, reply exactly:
   "Insufficient evidence in the corpus to answer this question."
4. If the question contains a false premise, correct it based on the records.
5. Be concise and factual. State uncertainty where it exists.
"""

def retrieve(question, k=5):
    qv = client.embeddings.create(model=EMBED_DEPLOYMENT, input=question).data[0].embedding
    res = col.query(query_embeddings=[qv], n_results=k)
    hits = []
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        hits.append({'ETid': meta['ETid'], 'country': meta['country'], 'sector': meta['sector'],
                     'year': meta['year'], 'similarity': round(1-dist,3), 'text': doc})
    return hits

def format_context(hits):
    return "\n\n".join(
        f"[ETid {h['ETid']}] ({h['country']}, {h['sector']}, {h['year']})\n{h['text']}"
        for h in hits)

def ask(question, k=5, show_sources=True):
    hits = retrieve(question, k)
    ctx = format_context(hits)
    resp = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[{"role":"system","content":SYSTEM_PROMPT},
                  {"role":"user","content":f"Retrieved records:\n\n{ctx}\n\nQuestion: {question}"}]
    )
    answer = resp.choices[0].message.content
    print("QUESTION:", question)
    print("\nANSWER:")
    print(textwrap.fill(answer, 100))
    if show_sources:
        print("\nRETRIEVED RECORDS:")
        for h in hits:
            print(f"  [ETid {h['ETid']}] sim={h['similarity']} | {h['country']}, {h['sector']}, {h['year']}")
    return {'question': question, 'answer': answer, 'hits': hits}

print('chatbot ready - use: ask("your question")')

chatbot ready - use: ask("your question")


In [8]:
def ask(question, k=5, show_sources=True):
    hits = retrieve(question, k)
    ctx = format_context(hits)
    resp = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[{"role":"system","content":SYSTEM_PROMPT},
                  {"role":"user","content":f"Retrieved records:\n\n{ctx}\n\nQuestion: {question}"}]
    )
    answer = resp.choices[0].message.content

    print("="*90)
    print("QUESTION:", question)
    print("="*90)
    print("\nANSWER:\n")
    for line in answer.split('\n'):
        print(textwrap.fill(line, 90) if line.strip() else "")

    if show_sources:
        print("\n" + "-"*90)
        print("RETRIEVED RECORDS")
        print("-"*90)
        for n, h in enumerate(hits, 1):
            print(f"\nRecord {n}  |  ETid {h['ETid']}  |  similarity {h['similarity']}")
            print(f"           {h['country']} · {h['sector']} · {h['year']}")
            print(textwrap.fill(h['text'][:300] + ('...' if len(h['text'])>300 else ''),
                                90, initial_indent='           ', subsequent_indent='           '))
        print("\n" + "="*90)
    return None      # <- stops the big dictionary dump

In [9]:
ask("What ransomware attacks affected hospitals?")

QUESTION: What ransomware attacks affected hospitals?

ANSWER:

The records show ransomware attacks affected the following healthcare providers:

- A Belgian hospital was hit by a ransomware attack that paralyzed parts of its computer
system and affected about 300,000 individuals [ETid 2521].
- Asl Napoli 3 Sud (Italy) suffered a ransomware attack using a virus that restricted
access to its database and affected data of 842,000 patients and employees [ETid 2080].
- A Polish healthcare facility suffered a ransomware attack that resulted in loss of
personal data; the controller was fined and failed to notify data subjects [ETid 2468].
- Centric Health Ltd. (Ireland) suffered a ransomware attack in which personal data of
about 70,000 people were accessed, altered and destroyed [ETid 1666].

Note: St. Olav’s Hospital (Norway) experienced multiple data leaks but the record does not
describe a ransomware attack [ETid 859].

--------------------------------------------------------------------

In [10]:
ask ("What happened in the Belgian DPA's case against a hospital in 2024, and what was the fine?")

QUESTION: What happened in the Belgian DPA's case against a hospital in 2024, and what was the fine?

ANSWER:

In 2024 the Belgian DPA investigated a hospital that had suffered a ransomware attack via
a server vulnerability, which paralyzed parts of its computer system and affected about
300,000 individuals; the DPA found the hospital had failed to carry out a data protection
impact assessment, lacked an adequate information security policy, and did not implement
appropriate technical and organizational measures (e.g., employee training and a process
for security updates) [ETid 2521]. The DPA fined the hospital EUR 200,000 [ETid 2521].

------------------------------------------------------------------------------------------
RETRIEVED RECORDS
------------------------------------------------------------------------------------------

Record 1  |  ETid 2521  |  similarity 0.768
           Belgium · Health Care · 2024
           The Belgian DPA has fined a hospital EUR 200,000. The hospi

In [11]:
ask("What was the average ransom amount paid by these organisations?")

QUESTION: What was the average ransom amount paid by these organisations?

ANSWER:

Insufficient evidence in the corpus to answer this question.

------------------------------------------------------------------------------------------
RETRIEVED RECORDS
------------------------------------------------------------------------------------------

Record 1  |  ETid 2561  |  similarity 0.511
           United Kingdom · Health Care · 2025
           The UK DPA (ICO) has fined Advanced Computer Software Group Ltd £3.07 million
           (EUR 3.5 million) for insufficient IT security (infringiment of Art. 32 (1) UK
           GDPR). The controller failed to implement appropriate technical and
           organisational measures to protect personal data. A ransomware attack in Au...

Record 2  |  ETid 2521  |  similarity 0.5
           Belgium · Health Care · 2024
           The Belgian DPA has fined a hospital EUR 200,000. The hospital had suffered a
           ransomware attack through a vul

In [12]:
ask("Why did the UK ICO fine Google 50 million euros for a ransomware attack?")

QUESTION: Why did the UK ICO fine Google 50 million euros for a ransomware attack?

ANSWER:

The premise is incorrect. The €50 million fine was imposed by the French data protection
authority (CNIL), not the UK ICO, and it related to Google’s lack of transparency,
insufficient information and lack of legal basis when a Google account was created during
Android phone setup — the CNIL found consents were not “specific” and “unambiguous” (Art.
5, Arts. 13/14 and Art. 6 GDPR; Art. 4(11) cited) [ETid 23].

------------------------------------------------------------------------------------------
RETRIEVED RECORDS
------------------------------------------------------------------------------------------

Record 1  |  ETid 2561  |  similarity 0.685
           United Kingdom · Health Care · 2025
           The UK DPA (ICO) has fined Advanced Computer Software Group Ltd £3.07 million
           (EUR 3.5 million) for insufficient IT security (infringiment of Art. 32 (1) UK
           GDPR). The 

## 6. The BM25 baseline
Same LLM, same prompt - only the RETRIEVAL differs (keyword instead of semantic).

In [15]:
def retrieve_bm25(question, k=5):
    scores = bm25.get_scores(question.lower().split())
    top = np.argsort(scores)[::-1][:k]
    hits = []
    for i in top:
        r = df.iloc[i]
        hits.append({'ETid': str(r['ETid']), 'country': str(r['Country']),
                     'sector': str(r['Sector']), 'year': 0,
                     'similarity': round(float(scores[i]),3), 'text': r['Summary']})
    return hits

def ask_bm25(question, k=5, show_sources=True):
    hits = retrieve_bm25(question, k)
    ctx = format_context(hits)
    resp = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[{"role":"system","content":SYSTEM_PROMPT},
                  {"role":"user","content":f"Retrieved records:\n\n{ctx}\n\nQuestion: {question}"}]
    )
    answer = resp.choices[0].message.content
    print("QUESTION:", question)
    print("\nBM25 BASELINE ANSWER:")
    print(textwrap.fill(answer, 100))
    if show_sources:
        print("\nRETRIEVED RECORDS:")
        for h in hits:
            print(f"  [ETid {h['ETid']}] score={h['similarity']} | {h['country']}, {h['sector']}")
    return {'question': question, 'answer': answer, 'hits': hits}

print('BM25 baseline ready - use: ask_bm25("your question")')

BM25 baseline ready - use: ask_bm25("your question")


In [16]:
q = "What happened when employees sent personal data to the wrong recipient?"
ask(q)
ask_bm25(q)

QUESTION: What happened when employees sent personal data to the wrong recipient?

ANSWER:

When employees sent personal data to the wrong recipient, personal data were disclosed to
unauthorised parties — including sensitive information about children and
criminal/private‑life data. Examples from the records:

- A contract containing an applicant’s name, address and telephone number was sent to the
wrong recipient [ETid 171].
- An employee’s payroll was sent to another employee, disclosing payroll data to an
unauthorised party [ETid 219].
- An employee of a public directorate posted nine letters to the wrong recipient that
contained personal data of 18 data subjects (including children, criminal data and data
related to private life); the unintended recipient notified the directorate five days
after posting, and the directorate notified the supervisory authority weeks later [ETid
251].
- A company emailed a customer containing another customer’s personal data due to
inadequate technica

{'question': 'What happened when employees sent personal data to the wrong recipient?',
 'answer': '- Sensitive information (including health records and invoices) was accidentally sent to wrong addressees due to mix-ups, causing personal data exposures [ETid 615; ETid 2054].  \n- In one case an email about COVID‑19 was sent to 198 recipients, exposing all recipients’ email addresses to each other [ETid 2054].  \n- Emails with customer personal data were mistakenly sent to incorrect recipients; the DPA found failures in technical and organizational measures in at least one case [ETid 2081].  \n- An email containing a customer’s name, postal address and insurance details was sent to the wrong recipient, and the controller did not notify the DPA or data subjects within 72 hours [ETid 741].  \n- In another case a wrong address entry led to a data subject receiving newsletters after requesting deletion; the controller replied with an unsubscribe link instead of deleting the data [ETid 813]

## 7. TRY IT ask anything!
Change the question and re-run. This is your live chatbot.

In [17]:
ask("What ransomware attacks affected hospitals?")

QUESTION: What ransomware attacks affected hospitals?

ANSWER:

Records show one explicit ransomware attack on a hospital and several on other healthcare
providers:

- Belgium — a hospital suffered a ransomware attack via a server vulnerability, affecting
about 300,000 individuals (fine EUR 200,000) [ETid 2521].
- Italy — ASL Napoli 3 Sud (a healthcare facility) suffered a ransomware attack that
affected data of 842,000 patients and employees (fine EUR 30,000) [ETid 2080].
- Poland — a healthcare facility suffered a ransomware attack resulting in loss of
personal data (fine EUR 9,200) [ETid 2468].
- Ireland — Centric Health Ltd. (a healthcare controller) suffered a ransomware attack
affecting ~70,000 people (fine EUR 460,000) [ETid 1666].

Note: the Norwegian case (St. Olav’s Hospital) involved multiple data leaks but is not
described as a ransomware attack in the records [ETid 859].

------------------------------------------------------------------------------------------
RETRIEVED R

In [18]:
ask("Which countries fined companies for weak security measures?")

QUESTION: Which countries fined companies for weak security measures?

ANSWER:

United Kingdom [ETid 2561]; Lithuania [ETid 1973]; Slovenia [ETid 3109].

------------------------------------------------------------------------------------------
RETRIEVED RECORDS
------------------------------------------------------------------------------------------

Record 1  |  ETid 2168  |  similarity 0.599
           France · Public Sector and Education · 2023
           Fine against municipality for lack of security measures (insufficient
           passwords)

Record 2  |  ETid 2561  |  similarity 0.574
           United Kingdom · Health Care · 2025
           The UK DPA (ICO) has fined Advanced Computer Software Group Ltd £3.07 million
           (EUR 3.5 million) for insufficient IT security (infringiment of Art. 32 (1) UK
           GDPR). The controller failed to implement appropriate technical and
           organisational measures to protect personal data. A ransomware attack in Au...

Re

In [19]:
# Test refusal behaviour (should say "insufficient evidence")
ask("What was the average ransom amount paid?")

QUESTION: What was the average ransom amount paid?

ANSWER:

Insufficient evidence in the corpus to answer this question.

------------------------------------------------------------------------------------------
RETRIEVED RECORDS
------------------------------------------------------------------------------------------

Record 1  |  ETid 2080  |  similarity 0.436
           Italy · Health Care · 2023
           The Italian DPA has fined Asl Napoli 3 Sud EUR 30,000.   The healthcare
           facility had suffered a ransomware attack that used a virus to restrict access
           to the healthcare facility's database and demanded a ransom to restore the
           functionality of its systems. During its investigation, the Garante DPA fo...

Record 2  |  ETid 2521  |  similarity 0.434
           Belgium · Health Care · 2024
           The Belgian DPA has fined a hospital EUR 200,000. The hospital had suffered a
           ransomware attack through a vulnerability in the server, whic

## 8. Side-by-side comparison (RAG vs BM25)
This is the core of RQ3 - run any question through both.

In [20]:
q = "What happened when employees sent personal data to the wrong recipient?"
print("="*70); print("RAG (semantic retrieval)"); print("="*70)
r1 = ask(q)
print("\n"+"="*70); print("BM25 (keyword retrieval)"); print("="*70)
r2 = ask_bm25(q)

RAG (semantic retrieval)
QUESTION: What happened when employees sent personal data to the wrong recipient?

ANSWER:

When employees (or the company) sent personal data to the wrong recipient, the data were
disclosed to unauthorized parties — for example, a payroll was sent to another employee
[ETid 219], and a contract containing name, address and telephone number was sent to the
wrong recipient [ETid 171]. In one case a Directorate employee mailed nine letters to the
wrong recipient containing personal data of 18 data subjects (including children, criminal
data and private-life information); the recipient told the Directorate five days later and
the Directorate notified the supervisory authority weeks after that [ETid 251]. Similar
incidents included a company email that delivered one customer’s data to another customer
due to inadequate security [ETid 263], and erroneous disclosures of children’s and family
contact/location data to unauthorized persons (including an alleged offender 

## 9. Your turn
Type any question below and run. Use this to explore before finalising
the 15-question benchmark.

In [ ]:
ask("YOUR QUESTION HERE")